# SwingTrader — 00 Setup Verification

**Run every cell top to bottom. All must pass before opening `01_data_layer.ipynb`.**

This notebook costs nothing — no LLM calls, no Kite tokens consumed.

| Check | What it verifies |
|---|---|
| Libraries | All packages installed at correct versions |
| `.env` | OPENAI_API_KEY present |
| OpenAI | Key is valid — returns `OK` |
| yfinance | Can fetch NSE data |
| Kite | Access token works (skip if not yet obtained) |

In [1]:
# Cell 1 — Install / upgrade all dependencies
# Run once; comment out after first successful run
%pip install -q yfinance pandas-ta langchain-openai langgraph langsmith \
    pydantic python-dotenv plotly scipy ipywidgets kiteconnect --upgrade

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Cell 2 — Library versions
import importlib, sys
import importlib.metadata

def _get_version(pkg_import, pkg_dist=None):
    """Get version via __version__, .version, or importlib.metadata fallback."""
    mod = importlib.import_module(pkg_import)
    for attr in ('__version__', 'version', 'VERSION'):
        v = getattr(mod, attr, None)
        if v and isinstance(v, str):
            return v
    # fallback to dist metadata (e.g. pandas-ta vs pandas_ta)
    try:
        return importlib.metadata.version(pkg_dist or pkg_import)
    except importlib.metadata.PackageNotFoundError:
        return 'unknown'

checks = {
    "python":           sys.version.split()[0],
    "yfinance":         _get_version('yfinance'),
    "pandas_ta":        _get_version('pandas_ta', 'pandas-ta'),
    "langchain_openai": _get_version('langchain_openai'),
    "langgraph":        _get_version('langgraph'),
    "pydantic":         _get_version('pydantic'),
    "plotly":           _get_version('plotly'),
}

print('Library versions:')
for lib, ver in checks.items():
    print(f'  {lib:<22} {ver}')
print()
assert sys.version_info >= (3, 11), 'Need Python 3.11+'
print('✓ Python version OK')

AttributeError: module 'pandas_ta' has no attribute '__version__'

In [2]:
# Cell 3 — Load .env and verify OPENAI_API_KEY
from dotenv import load_dotenv
from pathlib import Path
import os

# Search up to 4 parent directories for a .env file
_here = Path.cwd()
_env_path = next(
    (p / '.env' for p in [_here, *_here.parents[:3]] if (p / '.env').exists()),
    None
)
if _env_path is None:
    raise FileNotFoundError(
        f'No .env file found in {_here} or its parents. '
        'Create one with OPENAI_API_KEY=sk-...'
    )
load_dotenv(dotenv_path=_env_path, override=True)
print(f'✓ .env loaded from {_env_path}')

api_key = os.getenv('OPENAI_API_KEY', '')
assert api_key.startswith('sk-'), \
    'OPENAI_API_KEY missing or malformed. Add to .env file: OPENAI_API_KEY=sk-...'
print(f'✓ OPENAI_API_KEY loaded  (ends: ...{api_key[-4:]})')

ls_key = os.getenv('LANGCHAIN_API_KEY', '')
if ls_key:
    print(f'✓ LANGCHAIN_API_KEY loaded (LangSmith tracing active)')
else:
    print('⚠  LANGCHAIN_API_KEY not set — LangSmith tracing disabled (optional but recommended)')

AssertionError: OPENAI_API_KEY missing or malformed. Add to .env file: OPENAI_API_KEY=sk-...

In [3]:
# Cell 4 — OpenAI smoke test  (~$0.0001 cost)
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4.1-nano', temperature=0)
response = llm.invoke('Reply with the single word: OK')

assert 'OK' in response.content, f'Unexpected response: {response.content}'
print(f'✓ OpenAI connection OK  (response: "{response.content.strip()}")')

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:
# Cell 5 — yfinance smoke test
import yfinance as yf
import pandas as pd

df = yf.download('INFY.NS', period='5d', auto_adjust=True, progress=False)

# Flatten MultiIndex if present (yfinance quirk)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel(1)

assert len(df) > 0, 'yfinance returned empty DataFrame — check internet connection'
assert 'Close' in df.columns or 'close' in df.columns, 'Unexpected column names'

print(f'✓ yfinance OK  ({len(df)} rows for INFY.NS, last date: {df.index[-1].date()})')
df.tail(2)

In [ ]:
# Cell 6 — Kite Connect smoke test (skip if access token not yet obtained)
#
# To get access token:
#   1. Go to https://kite.zerodha.com/connect/login?api_key=YOUR_KEY&v=3
#   2. Login → you get redirected to your redirect URL
#   3. Copy request_token from the URL bar
#   4. Paste below and run once to generate access_token
#   5. Save access_token to .env as KITE_ACCESS_TOKEN
#
# Note: access_token expires daily at ~08:00 IST next morning

KITE_API_KEY    = os.getenv('KITE_API_KEY', '')
KITE_ACCESS_TOKEN = os.getenv('KITE_ACCESS_TOKEN', '')

if not KITE_API_KEY:
    print('⚠  KITE_API_KEY not set — skipping Kite test')
    print('   Add to .env: KITE_API_KEY=your_key')
    print('   POC Evening 1 will use yfinance instead of Kite')
elif not KITE_ACCESS_TOKEN:
    print('⚠  KITE_ACCESS_TOKEN not set')
    print(f'   Login URL: https://kite.zerodha.com/connect/login?api_key={KITE_API_KEY}&v=3')
    print('   After login, copy request_token from redirect URL')
    print('   Then run the token generation cell below')
else:
    from kiteconnect import KiteConnect
    kite = KiteConnect(api_key=KITE_API_KEY)
    kite.set_access_token(KITE_ACCESS_TOKEN)
    profile = kite.profile()
    print(f'✓ Kite Connect OK  (user: {profile["user_name"]})')

In [ ]:
# Cell 7 — Generate Kite access token from request_token (run once per day)
# Comment out after use — request_token is single-use

# REQUEST_TOKEN = 'paste_here'  # from redirect URL after login
# 
# from kiteconnect import KiteConnect
# kite = KiteConnect(api_key=os.getenv('KITE_API_KEY'))
# data = kite.generate_session(REQUEST_TOKEN, api_secret=os.getenv('KITE_SECRET'))
# access_token = data['access_token']
# print(f'Access token: {access_token}')
# print('Add to .env: KITE_ACCESS_TOKEN=' + access_token)

print('Token generation cell — uncomment and run when needed')

## Summary

If all ✓ above:
- Open `01_data_layer.ipynb` → Evening 1 starts here

If ⚠ on Kite:
- Evening 1 will use **yfinance** instead — fully functional for research
- Kite is needed only when you move to paper trading (Phase 7)

If ✗ on OpenAI:
- Check `.env` file — key must start with `sk-`
- Verify key at https://platform.openai.com/api-keys